# PCA

Principal Component Analysis replaces the columns of a dataset with new columns that are

1. **uncorrelated** with each other, and
2. **sorted** by how much variance they carry,

so that the first few of them keep most of the information and the rest can be dropped.

The algorithm has five steps. This notebook implements it, and we check it against
`sklearn`, and then reduces a real 768-dimensional dataset.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.decomposition import PCA

## The five steps

For data $X \in \mathbb{R}^{n \times d}$, one row per sample:

$$\mu = \frac{1}{n}\sum_{i=1}^{n} x_i \qquad \tilde{X} = X - 1\mu^T \qquad C = \frac{\tilde{X}^T \tilde{X}}{n - 1}$$

1. subtract the mean of every column
2. build the covariance matrix of the centred data
3. eigendecompose it, the eigenvectors are the new axes
4. sort them by eigenvalue, largest first
5. project the data onto the first `k` of them

In [ ]:
def pca_fit(X, k):
    mean = X.mean(axis=0)
    Xc = X - mean

    C = Xc.T @ Xc / (len(Xc) - 1)
    values, vectors = np.linalg.eigh(C)

    order = np.argsort(values)[::-1]
    values, vectors = values[order], vectors[:, order]

    return mean, values, vectors, Xc @ vectors[:, :k]


def pca_transform(X, mean, vectors, k):
    return (X - mean) @ vectors[:, :k]

## A real reduction: 768 features

1,000 STL-10 images, embedded with a multimodal model into 768 dimensions and spanning 10
classes. Reducing that to a couple of dozen numbers is a genuine application, not a toy.

In [ ]:
d = np.load("data/embeddinggemma_stl10_1000.npz")
E, y = d["E"], d["y"]

mean2, values2, vectors2, _ = pca_fit(E, 1)

ratio = values2 / values2.sum()
cumulative = np.cumsum(ratio)

for k in (2, 8, 16, 32, 64, 128):
    print(f"{k:4} components -> {cumulative[k - 1]:6.1%} of the variance")

In [ ]:
ref = PCA().fit(E)

print(f"{'k':>5}  {'by hand':>10}  {'sklearn':>10}  {'difference':>12}")
for k in (2, 8, 16, 32, 64, 128):
    mine = cumulative[k - 1]
    theirs = ref.explained_variance_ratio_[:k].sum()
    print(f"{k:5}  {mine:10.2%}  {theirs:10.2%}  {mine - theirs:12.2e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

axes[0].semilogy(values2[:60], marker=".")
axes[0].set_xlabel("component")
axes[0].set_ylabel("eigenvalue")
axes[0].set_title("the spectrum")

axes[1].plot(cumulative[:200] * 100, marker=".", ms=3)
axes[1].axhline(90, color="grey", ls="--", lw=1)
axes[1].set_xlabel("components kept")
axes[1].set_ylabel("% variance kept")
axes[1].set_title("cumulative")

plt.tight_layout()
plt.show()